In [11]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, recall_score
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier, XGBRegressor

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Create models directory
os.makedirs('../models', exist_ok=True)

# ==========================================
# PART 1: TRAIN THE MULTI-DISEASE SYSTEM (BRFSS)
# ==========================================
print("--- Loading 1 Million Row Dataset ---")
try:
    df = pd.read_csv('../data/processed/multi_year_health_data.csv')
except FileNotFoundError:
    df = pd.read_csv('data/processed/multi_year_health_data.csv')

# Define Features (Inputs) and Targets (Diseases)
X_columns = ['Age', 'BMI', 'Smoker', 'PhysicalHealthDays', 'MentalHealthDays', 
             'Income', 'Education', 'Avg_AQI']

targets = ['HeartAttack', 'Angina', 'Stroke', 'Asthma', 'SkinCancer', 
           'OtherCancer', 'COPD', 'Depression', 'KidneyDisease', 
           'Diabetes', 'Arthritis', 'DifficultyWalking']

print(f"Starting Training Pipeline (Logistic vs RF vs XGBoost) for {len(targets)} diseases...")

metrics_summary = []

# XGBoost Hyperparameter Grid (Optimized for speed/performance balance)
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.1, 0.01],
    'scale_pos_weight': [1] # Adjust if imbalance remains
}

for target in targets:
    print(f"\n[Training Module] Target: {target}")
    
    # 1. Prepare Data
    df_clean = df.dropna(subset=X_columns + [target])
    X = df_clean[X_columns]
    y = df_clean[target]
    
    # 2. Handle Imbalance (Crucial)
    rus = RandomUnderSampler(random_state=42)
    X_res, y_res = rus.fit_resample(X, y)
    
    X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)
    
    # --- MODEL 1: BASELINE (Logistic Regression) ---
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train, y_train)
    base_pred = log_reg.predict(X_test)
    base_acc = accuracy_score(y_test, base_pred)
    base_recall = recall_score(y_test, base_pred)

    # --- MODEL 2: CLASSICAL (Random Forest) ---
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_acc = accuracy_score(y_test, rf_pred)
    rf_recall = recall_score(y_test, rf_pred)

    # --- MODEL 3: ADVANCED (XGBoost with GridSearch) ---
    xgb = XGBClassifier(eval_metric='logloss', use_label_encoder=False, n_jobs=-1)
    grid_search = GridSearchCV(estimator=xgb, param_grid=xgb_param_grid, 
                               cv=3, scoring='recall', verbose=0, n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    best_xgb = grid_search.best_estimator_
    xgb_pred = best_xgb.predict(X_test)
    xgb_acc = accuracy_score(y_test, xgb_pred)
    xgb_recall = recall_score(y_test, xgb_pred)
    xgb_f1 = f1_score(y_test, xgb_pred)

    # Comparison Output
    print(f"   -> Logistic Acc: {base_acc:.2%} | Recall: {base_recall:.2%}")
    print(f"   -> Random Forest Acc: {rf_acc:.2%} | Recall: {rf_recall:.2%}")
    print(f"   -> XGBoost (Tuned) Acc: {xgb_acc:.2%} | Recall: {xgb_recall:.2%} | F1: {xgb_f1:.2f}")

    # Determine Winner (Prioritize Recall for Health, then Accuracy)
    # We save XGBoost by default as it's the "Advanced" model, unless RF is significantly better
    final_model = best_xgb
    
    # Save Metrics for Report
    metrics_summary.append({
        'Disease': target,
        'Baseline_Acc': base_acc,
        'RF_Acc': rf_acc,
        'XGB_Acc': xgb_acc,
        'XGB_Recall': xgb_recall,
        'Best_Params': str(grid_search.best_params_)
    })
    
    # Save the Best Model
    joblib.dump(final_model, f'../models/model_{target}.pkl')

# Save metrics table
pd.DataFrame(metrics_summary).to_csv('../models/training_metrics.csv', index=False)
print("\n--- Training Complete. Comparison Metrics Saved. ---")


# ==========================================
# PART 2: SPECIALIZED MODULES (Parkinson's & Autism)
# ==========================================
print("\n[Training Module] Parkinson's & Autism")

# --- 1. Parkinson's Disease ---
try:
    path_pk = '../data/Processed/parkinsons_clean.csv'
    if not os.path.exists(path_pk): path_pk = 'data/Processed/parkinsons_clean.csv'

    df_pk = pd.read_csv(path_pk)
    # Drop metadata columns to keep only vocal features
    X_pk = df_pk.drop(columns=['total_UPDRS', 'subject#', 'motor_UPDRS', 'test_time', 'age', 'sex'], errors='ignore')
    y_pk = df_pk['total_UPDRS']
    
    # Train Parkinson's Model
    model_pk = XGBRegressor(n_estimators=100, n_jobs=-1)
    model_pk.fit(X_pk, y_pk)
    
    joblib.dump(model_pk, '../models/model_parkinsons.pkl')
    print("   -> Parkinson's Model Saved.")
except Exception as e:
    print(f"   -> Parkinson's Error: {e}")

# --- 2. Autism Screening (FIXED FOR HEADERLESS DATA) ---
try:
    path_aut = '../data/Processed/autism_clean.csv'
    if not os.path.exists(path_aut): path_aut = 'data/Processed/autism_clean.csv'
        
    df_aut = pd.read_csv(path_aut)
    
    # FIX: Check if headers are missing (e.g., '0', '1', '2'...)
    if df_aut.columns[0] == '0':
        print("   -> Detected headerless CSV. Using LAST column as Target.")
        # Assign temporary names: Feat1, Feat2... Target
        new_cols = [f"Feat_{i}" for i in range(len(df_aut.columns)-1)] + ['Target']
        df_aut.columns = new_cols
        target = 'Target'
    else:
        # Search for text headers
        target_col = [c for c in df_aut.columns if 'class' in c.lower() or 'asd' in c.lower()]
        target = target_col[0] if target_col else df_aut.columns[-1] # Fallback to last col

    # Prepare Data
    X_aut = df_aut.drop(columns=[target])
    X_aut = pd.get_dummies(X_aut, drop_first=True)
    
    # Clean Target (Convert 'YES'/'NO' or 'p'/'n' to 1/0)
    y_aut = df_aut[target].astype(str).apply(lambda x: 1 if x.lower() in ['yes', '1', 'p', 'positive', 'class1'] else 0)
    
    # Train Autism Model (Silent Mode)
    model_aut = XGBClassifier(eval_metric='logloss', n_jobs=-1)
    model_aut.fit(X_aut, y_aut)
    
    # Save Model & Feature List (Critical for UI)
    joblib.dump(model_aut, '../models/model_autism.pkl')
    joblib.dump(X_aut.columns.tolist(), '../models/autism_features.pkl')
    
    print(f"   -> Autism Model Saved. (Target Column: {target})")

except Exception as e:
    print(f"   -> Autism Error: {e}")

print("\nAll Systems Go! Ready to run 'streamlit run src/app.py'")

--- Loading 1 Million Row Dataset ---
Starting Training Pipeline (Logistic vs RF vs XGBoost) for 12 diseases...

[Training Module] Target: HeartAttack


/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:17:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:17:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:17:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:17:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packa

   -> Logistic Acc: 69.74% | Recall: 76.10%
   -> Random Forest Acc: 71.08% | Recall: 80.97%
   -> XGBoost (Tuned) Acc: 70.55% | Recall: 80.87% | F1: 0.73

[Training Module] Target: Angina


/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:17:34] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:17:34] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:17:34] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:17:34] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packa

   -> Logistic Acc: 71.11% | Recall: 78.07%
   -> Random Forest Acc: 72.39% | Recall: 83.57%
   -> XGBoost (Tuned) Acc: 71.67% | Recall: 82.33% | F1: 0.74

[Training Module] Target: Stroke


...

   -> Logistic Acc: 61.75% | Recall: 60.68%
   -> Random Forest Acc: 63.87% | Recall: 61.46%
   -> XGBoost (Tuned) Acc: 64.79% | Recall: 65.13% | F1: 0.65

[Training Module] Target: Arthritis


/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:18:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:18:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:18:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:18:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packa

   -> Logistic Acc: 70.84% | Recall: 75.16%
   -> Random Forest Acc: 71.80% | Recall: 80.03%
   -> XGBoost (Tuned) Acc: 71.09% | Recall: 79.43% | F1: 0.73

[Training Module] Target: DifficultyWalking


/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:19:30] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:19:30] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:19:30] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [03:19:30] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/opt/anaconda3/lib/python3.12/site-packa

   -> Logistic Acc: 74.38% | Recall: 75.40%
   -> Random Forest Acc: 79.51% | Recall: 80.69%
   -> XGBoost (Tuned) Acc: 79.76% | Recall: 81.05% | F1: 0.80

--- Training Complete. Comparison Metrics Saved. ---

[Training Module] Parkinson's & Autism
   -> Parkinson's Model Saved.
   -> Detected headerless CSV. Using LAST column as Target.
   -> Autism Model Saved. (Target Column: Target)

All Systems Go! Ready to run 'streamlit run src/app.py'
